# LAB 2 — Medical Image Segmentation
**Medical Image Analysis · Biomedical Engineering · URJC 2026/2027**
**Group:** TODO · **Authors:** TODO

| Part | Content |
|---|---|
| **Part 1** | Thresholding, region growing and watershed · parameter influence · comparison and choice per ROI · pre-/post-processing |
| **Part 2** | Active contours (snakes) |
| **Part 3** | Additional state-of-the-art segmentation method |

In [ ]:
# --- Setup: libraries, project modules and reproducibility ---------------------------
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Project root = folder that contains src/ (works whether the notebook runs from notebooks/ or the root)
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src").is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

from src import (utils, export_seg, metrics, preprocessing, thresholding, region_growing,
                 watershed_seg, active_contours)

utils.ensure_dirs()          # creates results/figures, results/tables, ...
SEED = utils.set_seed(42)    # reproducibility

# PART 1 — Thresholding, Region Growing and Watershed

In [ ]:
# --- Step 2: load the two Lab 1 images and define the ROI of each one ------------------
# ROI = bounding box (ymin, ymax, xmin, xmax) in pixels -> img[ymin:ymax, xmin:xmax]
IMAGES = {
    "brain": {"file": "Brain_MRI_131749_T1.png",
              "structure": "Hypointense lesion (frontal white matter)",
              "roi": (120, 205, 125, 215)},
    "chest": {"file": "Chest_Xray_PA_3-8-2010.png",
              "structure": "Right lung",
              "roi": (140, 1910, 130, 1130)},
}

images, rois = {}, {}
for key, info in IMAGES.items():
    images[key] = utils.load_image(info["file"])          # 2-D grayscale image, float in [0, 1]
    rois[key] = utils.crop_roi(images[key], info["roi"])  # sub-image that will be segmented
    h, w, n_roi = utils.roi_dimensions(info["roi"])
    print(f"{key}: image {images[key].shape} -> ROI {info['roi']} = {h} x {w} px (N_ROI = {n_roi})")

    # Full image with the ROI in red vs. cropped ROI
    fig = utils.plot_roi_overview(images[key], info["roi"], title=info["structure"])
    utils.save_figure(fig, f"part1_roi_{key}.png")
    plt.show()

In [ ]:
# --- Step 3: manual reference segmentations (ground truth) made in 3D Slicer -----------
GT_FILES = {"brain": "SegmentationTumor.nrrd", "chest": "SegmentationLung.nrrd"}

gts_full, gts = {}, {}
for key, info in IMAGES.items():
    # Slicer may export only the box around the segment -> place it in the full image grid
    gts_full[key] = export_seg.read_mask(utils.DATA_GT / GT_FILES[key], reference_shape=images[key].shape)
    gts[key] = utils.crop_roi(gts_full[key], info["roi"])   # same ROI as the image
    assert gts[key].sum() == gts_full[key].sum(), f"{key}: part of the reference lies outside the ROI"

    # Binary mask with the size of the full image (0 = background, 255 = structure)
    mask_path = export_seg.write_mask_png(gts_full[key], utils.DATA_GT / f"{key}_reference_mask.png")
    print(f"{key}: reference = {gts_full[key].sum()} px ({100 * gts[key].mean():.1f} % of the ROI) "
          f"-> {mask_path.relative_to(PROJECT_ROOT)}")

# Binary reference masks (white = structure, black = background): full image (ROI in red) and ROI
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
for row, key in zip(axes, IMAGES):
    ymin, ymax, xmin, xmax = IMAGES[key]["roi"]
    row[0].imshow(gts_full[key], cmap="gray", vmin=0, vmax=1)
    row[0].add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, fill=False, edgecolor="red"))
    row[0].set_title(f"{key}: binary mask {gts_full[key].shape[0]}×{gts_full[key].shape[1]}")
    row[1].imshow(gts[key], cmap="gray", vmin=0, vmax=1)
    row[1].set_title(f"{key}: binary mask in the ROI ({gts[key].sum()} px)")
    for ax in row:
        ax.axis("off")
fig.tight_layout()
utils.save_figure(fig, "part1_reference.png")
plt.show()

In [ ]:
# --- Step 4: THRESHOLDING — sweep of T, Dice against the reference, best threshold -----
# In both ROIs the structure is darker than its surroundings -> mask = pixels with I <= T
seg_masks = {key: {} for key in IMAGES}     # best mask of each method (filled in steps 4-6)
best_params = {key: {} for key in IMAGES}   # best parameter(s) of each method

fig, axes = plt.subplots(1, 2, figsize=(10, 6))
for ax, key in zip(axes, IMAGES):
    roi_img, gt = rois[key], gts[key]

    T_otsu = thresholding.otsu(roi_img)                             # automatic threshold (Otsu)
    T_values = np.linspace(*np.percentile(roi_img, [1, 99]), 100)   # thresholds to test
    dices = [metrics.dice(thresholding.apply_threshold(roi_img, high=T), gt) for T in T_values]
    T_best = float(T_values[np.argmax(dices)])                      # T with the highest Dice
    dice_otsu = metrics.dice(thresholding.apply_threshold(roi_img, high=T_otsu), gt)

    mask = thresholding.apply_threshold(roi_img, high=T_best)
    seg_masks[key]["Thresholding"] = mask
    best_params[key]["Thresholding"] = {"T": round(T_best, 3)}
    print(f"{key}: Otsu T = {T_otsu:.3f} (Dice = {dice_otsu:.3f}) | best T = {T_best:.3f} (Dice = {max(dices):.3f})")

    # Binary mask with the best threshold
    ax.imshow(mask, cmap="gray")
    ax.set_title(f"{key}: binary mask (T = {T_best:.3f}, Dice = {max(dices):.3f})")
    ax.axis("off")
fig.tight_layout()
utils.save_figure(fig, "part1_thresholding.png")
plt.show()

In [ ]:
# --- Step 5: REGION GROWING — sweep of tolerance and connectivity, Dice, best parameters --
# Seeds = pixels inside the structure, chosen by visual inspection, in full-image (row, col) coordinates
SEEDS = {
    "brain": [(162, 168)],                            # dark centre of the lesion
    "chest": [(700, 600), (1000, 550), (1300, 700)],  # upper, middle and lower lung
}
TOLERANCES = np.round(np.arange(0.01, 0.41, 0.01), 2)   # max |I - mean(seeds)| to join the region

fig, axes = plt.subplots(1, 2, figsize=(10, 6))
for ax, key in zip(axes, IMAGES):
    roi_img, gt = rois[key], gts[key]
    seeds = utils.to_roi_coords(SEEDS[key], IMAGES[key]["roi"])   # seeds in ROI coordinates

    # Dice of every (tolerance, connectivity) pair
    results = pd.DataFrame([
        {"connectivity": conn, "tol": tol,
         "Dice": metrics.dice(region_growing.region_growing(roi_img, seeds, tol, connectivity=conn), gt)}
        for conn in (4, 8) for tol in TOLERANCES])
    for conn in (4, 8):
        res = results[results["connectivity"] == conn]
        best_c = res.loc[res["Dice"].idxmax()]
        print(f"{key}: {conn}-connectivity -> best tol = {best_c['tol']:.2f} (Dice = {best_c['Dice']:.3f})")

    best = results.loc[results["Dice"].idxmax()]                     # best pair overall
    tol_best, conn_best = float(best["tol"]), int(best["connectivity"])
    mask = region_growing.region_growing(roi_img, seeds, tol_best, connectivity=conn_best)
    seg_masks[key]["Region growing"] = mask
    best_params[key]["Region growing"] = {"tol": tol_best, "connectivity": conn_best}

    # Binary mask with the best parameters (red crosses = seeds)
    ax.imshow(mask, cmap="gray")
    ax.plot([c for _, c in seeds], [r for r, _ in seeds], "r+", markersize=12, markeredgewidth=2)
    ax.set_title(f"{key}: binary mask (tol = {tol_best:.2f}, {conn_best}-conn., Dice = {best['Dice']:.3f})")
    ax.axis("off")
fig.tight_layout()
utils.save_figure(fig, "part1_region_growing.png")
plt.show()

# PART 2 — Active Contours (Snakes)

## 2.1 The snake (classical active contour)
A **snake** is a closed curve, defined by a set of points, that moves over the image until it minimises an energy

Implementation: `skimage.segmentation.active_contour` (wrapped in `src/active_contours.py`). Here $w_{line}=0$, $w_{edge}=1$, $\gamma=0.001$ and up to 2500 iterations; we study the influence of $\alpha$ and $\beta$.

## 2.2 Initial contour of each ROI
The snake is a *local* method: it needs an initial curve placed **around** the structure, which then shrinks until the edges stop it.

In [ ]:
# --- 2.2: initial contour of each ROI --------------------------------------------------
# Initial ellipse in full-image (row, col) coordinates, like the ROI, chosen by visual inspection.
# scale = downsampling factor of the ROI before running the snake:
#   - brain (85 x 90 px): 1, the snake runs on the original ROI.
#   - chest (1770 x 1000 px): 8. At full resolution the edge force is too weak compared with the
#     elasticity and the snake collapses to a point; on the reduced image the lung border is
#     stronger and the snake also runs much faster.
# sigma = Gaussian smoothing before the snake (larger sigma -> edges felt from further away,
#         but small structures such as the lesion get blurred and lost).
SNAKE_CFG = {
    "brain": {"center": (161, 169), "radii": (16, 16), "scale": 1, "sigma": 1},
    "chest": {"center": (1025, 630), "radii": (840, 480), "scale": 8, "sigma": 2},
}

snake_inits = {}   # initial curve of each ROI, in ROI coordinates
for key in IMAGES:
    center = utils.to_roi_coords([SNAKE_CFG[key]["center"]], IMAGES[key]["roi"])[0]   # full image -> ROI
    snake_inits[key] = active_contours.ellipse_init(center, SNAKE_CFG[key]["radii"])

# Initial contour (blue) on each ROI, with the reference contour (green) for orientation
fig, axes = plt.subplots(1, 2, figsize=(10, 6))
for ax, key in zip(axes, IMAGES):
    init = snake_inits[key]
    ax.imshow(rois[key], cmap="gray")
    ax.contour(gts[key], levels=[0.5], colors="lime", linewidths=1)
    ax.plot(init[:, 1], init[:, 0], "--b", linewidth=2)
    ax.set_title(f"{key}: initial snake ({len(init)} points)")
    ax.axis("off")
fig.tight_layout()
utils.save_figure(fig, "part2_snake_init.png")
plt.show()

## 2.3 Influence of the parameters ($\alpha$, $\beta$)
Grid of $\alpha$ (elasticity) × $\beta$ (rigidity). Blue = initial curve, red = final snake, green = reference.

In [ ]:
# --- 2.3: sweep of alpha (elasticity) and beta (rigidity), Dice against the reference ----
ALPHAS = [0.001, 0.005, 0.015, 0.05]
BETAS = [0.1, 1, 10]

snake_results = {}   # one DataFrame per image with the Dice of every (alpha, beta)
for key in IMAGES:
    rows = []
    height, width = rois[key].shape
    fig, axes = plt.subplots(len(BETAS), len(ALPHAS),
                             figsize=(3 * len(ALPHAS), 3 * len(BETAS) * height / width),
                             layout="constrained")
    init = snake_inits[key]
    for i, beta in enumerate(BETAS):
        for j, alpha in enumerate(ALPHAS):
            snake, mask = active_contours.snake_segmentation(
                rois[key], init, alpha, beta, sigma=SNAKE_CFG[key]["sigma"], scale=SNAKE_CFG[key]["scale"])
            d = metrics.dice(mask, gts[key])
            rows.append({"alpha": alpha, "beta": beta, "Dice": d, "Region size (px)": int(mask.sum())})

            ax = axes[i, j]
            ax.imshow(rois[key], cmap="gray")
            ax.contour(gts[key], levels=[0.5], colors="lime", linewidths=1)
            ax.plot(init[:, 1], init[:, 0], "--b", linewidth=1)
            ax.plot(snake[:, 1], snake[:, 0], "-r", linewidth=1.5)
            ax.set_title(f"α={alpha}, β={beta}\nDice={d:.3f}", fontsize=9)
            ax.axis("off")
    fig.suptitle(f"{key}: snake for different α (columns) and β (rows)")
    utils.save_figure(fig, f"part2_snake_params_{key}.png")
    plt.show()

    snake_results[key] = pd.DataFrame(rows)
    dice_table = snake_results[key].pivot(index="beta", columns="alpha", values="Dice")   # rows = beta, cols = alpha
    utils.save_table(dice_table, f"part2_snake_dice_{key}.csv")
    print(f"{key}: Dice of the snake (rows = beta, columns = alpha)")
    display(dice_table.round(3))

## 2.4 Best snake and comparison with Part 1
The best $(\alpha, \beta)$ of each ROI is kept and compared with the Part 1 methods stored in `seg_masks` (same ROI, same reference).

In [ ]:
# --- 2.4: best snake of each ROI and comparison with the Part 1 methods -------------------
for key in IMAGES:
    results = snake_results[key]
    best = results.loc[results["Dice"].idxmax()]          # row with the highest Dice
    snake, mask = active_contours.snake_segmentation(rois[key], snake_inits[key], best["alpha"], best["beta"],
                                                     sigma=SNAKE_CFG[key]["sigma"], scale=SNAKE_CFG[key]["scale"])
    seg_masks[key]["Snakes"] = mask
    best_params[key]["Snakes"] = {"alpha": best["alpha"], "beta": best["beta"], "sigma": SNAKE_CFG[key]["sigma"]}
    print(f"{key}: best snake alpha = {best['alpha']}, beta = {best['beta']} (Dice = {best['Dice']:.3f})")

    # Metrics of every method obtained so far against the reference segmentation
    comparison = metrics.evaluate_methods(seg_masks[key], gts[key])
    utils.save_table(comparison, f"part2_comparison_{key}.csv")
    display(comparison.round(3))

    # Contour of each method (red) on the ROI, reference dashed in green
    fig = utils.show_overlays(rois[key], seg_masks[key], gt=gts[key], title=f"{key}: Part 1 methods vs. snakes")
    utils.save_figure(fig, f"part2_comparison_{key}.png")
    plt.show()

# PART 3 — Other Segmentation Methods